In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Cài đặt thư viện (với TesseractOCR cần tải dữ liệu train cho tiếng Việt)

In [8]:
import os
!pip install pdf2image pytesseract opencv-python-headless -q
!apt-get update -qq
!apt-get install -y tesseract-ocr poppler-utils -q
tessdata_path = '/usr/share/tesseract-ocr/4.00/tessdata'
os.makedirs(tessdata_path, exist_ok=True)
vie_data_url = 'https://github.com/tesseract-ocr/tessdata/raw/main/vie.traineddata'
target_file = os.path.join(tessdata_path, 'vie.traineddata')
if not os.path.exists(target_file):
    !wget -O {target_file} {vie_data_url}

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


# 2. Cấu hình đường dẫn

In [9]:
import os
VOLUME = 1
FIRST_PAGE_NUMBER = 22

BASE_OUTPUT = f'/content/drive/MyDrive/THESIS/OCR/VOL{VOLUME}'
TEXT_OUT_DIR = os.path.join(BASE_OUTPUT, 'texts')
IMAGE_OUT_DIR = os.path.join(BASE_OUTPUT, 'images')
pdf_filename = os.path.join(BASE_OUTPUT, f'vol{VOLUME}.pdf')

os.makedirs(TEXT_OUT_DIR, exist_ok=True)
os.makedirs(IMAGE_OUT_DIR, exist_ok=True)
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata'

print(f'\nVolume: {VOLUME}\nSource PDF: {pdf_filename}\nOutput Texts: {TEXT_OUT_DIR}\nOutput Images: {IMAGE_OUT_DIR}\n')


Volume: 1
Source PDF: /content/drive/MyDrive/THESIS/OCR/VOL1/vol1.pdf
Output Texts: /content/drive/MyDrive/THESIS/OCR/VOL1/texts
Output Images: /content/drive/MyDrive/THESIS/OCR/VOL1/images



# 3. Định nghĩa hàm xử lý ảnh và OCR

In [10]:
import cv2
import numpy as np
import pytesseract
from PIL import Image

def xu_ly_anh(img_path):
    img = cv2.imread(img_path)
    if img is None: return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Tự động cắt lề
    _, thresh = cv2.threshold(img, 15, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]

    # Khử nhiễu và làm nét
    img = cv2.fastNlMeansDenoising(img, h=3)
    kernel = np.array([[0, -0.5, 0], [-0.5, 3, -0.5], [0, -0.5, 0]])
    img = cv2.filter2D(img, -1, kernel)

    # Chuyển đen trắng (Otsu)
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return img

def chay_ocr(img_ready):
    config = r'--psm 1'
    return pytesseract.image_to_string(img_ready, lang='vie', config=config).strip()

# 4. Chuyển đổi PDF thành ảnh

In [ ]:
from tqdm import tqdm
from pdf2image import convert_from_path, pdfinfo_from_path

try:
    os.makedirs(IMAGE_OUT_DIR, exist_ok=True)

    info = pdfinfo_from_path(pdf_filename)
    max_pdf_pages = info['Pages']
    print(f'Tổng số trang: {max_pdf_pages}')

    for pdf_idx in tqdm(range(1, max_pdf_pages + 1), desc='Đang lưu ảnh'):
        book_page = FIRST_PAGE_NUMBER + (pdf_idx - 1)
        base_name = f'vol{VOLUME}_p{book_page:04d}'
        img_path = os.path.join(IMAGE_OUT_DIR, f'{base_name}.png')

        if os.path.exists(img_path): continue

        images = convert_from_path(
            pdf_filename, dpi=300,
            first_page=pdf_idx, last_page=pdf_idx,
            use_pdftocairo=True, use_cropbox=True
        )

        if images:
            images[0].save(img_path, 'PNG')
            del images

except Exception as e:
    print(f'Lỗi: {e}')

Tổng số trang: 978


Đang lưu ảnh: 100%|██████████| 978/978 [39:14<00:00,  2.41s/it]


# 5. Chạy OCR

In [11]:
from tqdm import tqdm
os.makedirs(TEXT_OUT_DIR, exist_ok=True)
files = sorted([f for f in os.listdir(IMAGE_OUT_DIR) if f.endswith('.png')])

print(f"Bắt đầu OCR {len(files)} trang...")

for img_file in tqdm(files, desc='Đang xử lý text'):
    name = os.path.splitext(img_file)[0]
    txt_path = os.path.join(TEXT_OUT_DIR, f'{name}.txt')
    full_img_path = os.path.join(IMAGE_OUT_DIR, img_file)

    if os.path.exists(txt_path): continue

    try:
        img_processed = xu_ly_anh(full_img_path)
        if img_processed is not None:
            kq = chay_ocr(img_processed)
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(kq)
    except Exception as e:
        print(f"Lỗi file {img_file}: {e}")

Bắt đầu OCR 978 trang...


Đang xử lý text: 100%|██████████| 978/978 [16:56<00:00,  1.04s/it]
